# SALI PyTorch Reproduction

Colab-ready practical reproduction of the SALI signal-to-image model.

In [ ]:
%pip install -q -e .

## Configuration

In [ ]:
from pathlib import Path
from sali.config import practical_config

cfg = practical_config(field='low')
cfg.output_dir = Path('runs/notebook-practical-low')
cfg.training.device = 'auto'
cfg.training.max_epochs = 1
cfg.training.batch_size = 4
cfg.data.train_samples = 64
cfg.data.val_samples = 16
cfg.data.test_samples = 16
cfg.output_dir.mkdir(parents=True, exist_ok=True)
cfg


## Generate training, validation and testing dataset

In [ ]:
from sali.data import generate_splits

splits, stats = generate_splits(cfg)
len(splits.train), len(splits.val), len(splits.test), stats


## Generated spectra

In [ ]:
from IPython.display import Image, display
from sali.plots import plot_spectra

sample = splits.test[0]
path = cfg.output_dir / 'figures' / 'generated_spectra.png'
plot_spectra(sample.raw_signals, path)
display(Image(filename=str(path)))


## True heatmap

In [ ]:
from sali.plots import plot_heatmap

path = cfg.output_dir / 'figures' / 'true_heatmap.png'
plot_heatmap(sample.heatmap, 'True heatmap', path)
display(Image(filename=str(path)))


## Train PyTorch SALI model

In [ ]:
from sali.train import train_model

result = train_model(cfg, splits)
result.best_checkpoint


## Training and validation loss

In [ ]:
from sali.plots import plot_loss

path = cfg.output_dir / 'figures' / 'loss.png'
plot_loss(result.history, path)
display(Image(filename=str(path)))


## Predicted and post-processed heatmaps

In [ ]:
import numpy as np
import torch
from sali.postprocess import postprocess_heatmap

device = next(result.model.parameters()).device
result.model.eval()
with torch.no_grad():
    pred = result.model(
        torch.from_numpy(sample.signals[0:1]).unsqueeze(0).to(device),
        torch.from_numpy(sample.signals[1:2]).unsqueeze(0).to(device),
    ).cpu().numpy()[0]
predictions = postprocess_heatmap(pred, cfg.data, cfg.model, cfg.postprocess)
pred_path = cfg.output_dir / 'figures' / 'predicted_heatmap.png'
plot_heatmap(pred, 'Predicted heatmap', pred_path)
display(Image(filename=str(pred_path)))
post = np.zeros_like(pred)
for item in predictions:
    row = int(round(item.row))
    col = int(round(item.col))
    post[0, max(0, row - 2):row + 3, max(0, col - 2):col + 3] = 1.0
post_path = cfg.output_dir / 'figures' / 'postprocessed_heatmap.png'
plot_heatmap(post, 'Post-processed heatmap', post_path)
display(Image(filename=str(post_path)))
[(p.az_khz, p.aperp_khz, p.confidence) for p in predictions[:10]]


## Original vs identified C13 reconstructed signals

In [ ]:
from dataclasses import replace
from sali.physics import Couplings, generate_sample_signals
from sali.plots import plot_signal_overlay

pred_couplings = Couplings(
    az_khz=np.array([item.az_khz for item in predictions], dtype=np.float32),
    aperp_khz=np.array([item.aperp_khz for item in predictions], dtype=np.float32),
)
clean_physics = replace(cfg.physics, add_shot_noise=False)
reconstructed = generate_sample_signals(
    pred_couplings,
    clean_physics,
    np.random.default_rng(cfg.data.seed + 202),
)
path = cfg.output_dir / 'figures' / 'signal_overlay.png'
plot_signal_overlay(sample.raw_signals, reconstructed, path)
display(Image(filename=str(path)))


## Precision and recall

In [ ]:
from sali.plots import plot_precision_recall
from sali.train import evaluate_model

metrics = evaluate_model(result.model, cfg, splits.test, max_samples=16)
path = cfg.output_dir / 'figures' / 'precision_recall.png'
plot_precision_recall(metrics, path)
display(Image(filename=str(path)))


## MAE

In [ ]:
from sali.plots import plot_mae

path = cfg.output_dir / 'figures' / 'mae.png'
plot_mae(metrics, path)
display(Image(filename=str(path)))
